### Track C - Integrate Pybullet simulation with Pytorch model

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader, random_split

import pybullet as p
import pybullet_data

import matplotlib.pyplot as plt
import math
import time
from pathlib import Path
from PIL import Image
import numpy as np
import h5py


pybullet build time: Nov 28 2023 23:45:17


### C1. Simplify the task drastically

1. Reach task

In [46]:
SEED=5
torch.manual_seed(SEED)
np.random.seed(SEED)
p.disconnect()

numActiveThreads = 0
stopping threads
destroy semaphore
semaphore destroyed
Thread with taskId 0 exiting
Thread TERMINATED
destroy main semaphore
main semaphore destroyed
finished
numActiveThreads = 0
btShutDownExampleBrowser stopping threads
Thread with taskId 0 exiting
Thread TERMINATED
destroy semaphore
semaphore destroyed
destroy main semaphore
main semaphore destroyed


In [72]:
PANDA_ARM_JOINTS = [0, 1, 2, 3, 4, 5, 6]
PANDA_FINGER_JOINTS = [9, 10]


class MiniArmEnv():
    def __init__(self, max_steps: int = 300, render: bool = True):
        self.max_steps = max_steps
        self.physics_engine_update = 240 # 240 Hz.
        self.success_threshold = 0.25
        self._step_count = 0
        
        mode = p.GUI if render else p.DIRECT
        self.client = p.connect(mode)

        # load scene
        self._load_scene()

        # get end effector index.
        self.end_effector = _get_end_effector(robotId=self.robotId)

    def reset(self):
        home_positions = [0, -0.785, 0, -2.356, 0, 1.571, 0.785]
        finger_home = [0.04, 0.04]  # open gripper

        for j, angle in zip(PANDA_ARM_JOINTS, home_positions):
            p.resetJointState(self.robotId, j, angle, targetVelocity=0)

        for j, angle in zip(PANDA_FINGER_JOINTS, finger_home):
            p.resetJointState(self.robotId, j, angle, targetVelocity=0)

        # reset sphere position
        p.resetBasePositionAndOrientation(bodyUniqueId=self.sphereId,
                                          posObj=[np.random.uniform(-0.4, 0.4),
                                                  np.random.uniform(-0.4, 0.4),
                                                  np.random.uniform(0.4, 0.4)],
                                          ornObj=[0.0, 0.0, 0.0, 1.0])
        
        self.sphereId_position = np.array(p.getBasePositionAndOrientation(self.sphereId)[0])
        
        p.stepSimulation()
        return self._get_obs()

    def step(self, action):
        
        self._apply_action(action)

        p.setTimeStep(1 / self.physics_engine_update)

        for _ in range(12):
            # TODO: add a break. If the EE reached the target position it stops. Use self._check_ee_close_sphere() every N steps.
            p.stepSimulation()

        self._step_count += 1        
                  
        obs = self._get_obs()
        reward = None
        done = self._is_done() or self._step_count >= self.max_steps
        info = {"step": self._step_count, "success": self._check_position_success()}

        return obs, reward, done, info

    def close(self):
        p.disconnect(self.client)


    # ── scene loading ────────────────────────────────────────────────────────
    def _load_scene(self):
        "Load plane, frank arm and sphere"
        
        p.setAdditionalSearchPath(pybullet_data.getDataPath())
        p.setGravity(0, 0, -9.81)

        p.loadURDF("plane.urdf")
        # upload the franka arm.
        self.robotId = p.loadURDF("franka_panda/panda.urdf",
                                basePosition=[0,0,0],
                                useFixedBase=True)

        # create a sphere
        radius = 0.1
        visual_shape = p.createVisualShape(shapeType=p.GEOM_SPHERE,
                                        radius=radius,
                                        rgbaColor=[0, 1, 0, 1])

        collision_shape = p.createCollisionShape(shapeType=p.GEOM_SPHERE,
                                                radius=radius)
                                                
        self.sphereId = p.createMultiBody(baseMass=0,
                                    baseCollisionShapeIndex=collision_shape,
                                    baseVisualShapeIndex=visual_shape,
                                    basePosition=[1,1,1])

    # ── action application ───────────────────────────────────────────────────

    def _apply_action(self, action):
        sphere_pos = self.sphereId_position # reach task: the end effector must reach the position of the sphere.
        delta_pos = action # delta (x,y,z) in meters (MKS system).
        target_pos = sphere_pos + delta_pos
        
        # inverse kinematics
        joint_angles = p.calculateInverseKinematics(
            bodyUniqueId=self.robotId,
            endEffectorLinkIndex=self.end_effector,
            targetPosition=target_pos, # TODO: Add the ee_orn???
            maxNumIterations=400,
            residualThreshold=1e-7
        )

        # move the arm
        p.setJointMotorControlArray(
            bodyIndex=self.robotId,
            jointIndices=PANDA_ARM_JOINTS + PANDA_FINGER_JOINTS,
            controlMode=p.POSITION_CONTROL,
            targetPositions = joint_angles,
            forces = [500]*len(PANDA_ARM_JOINTS + PANDA_FINGER_JOINTS)
        )


    # ── observations ─────────────────────────────────────────────────────────

    def _get_obs(self):
        rgb_arr, depth_arr = self._render_camera()
        proprio = self._get_proprio()
        ee_pos, ee_orn = self._get_ee_pose() # position of robotic arm end effector in the URDF link frame (the one placed at a meaningful reference point by the URDF designer)
        sphere_pos = self.sphereId_position
        
        return {
            "rgb": rgb_arr,
            "depth": depth_arr,
            "proprio": proprio,
            "ee_pos": ee_pos,
            "ee_orn": ee_orn,
            "sphere_pos": sphere_pos
        }

    def _render_camera(self):
        view_mat = p.computeViewMatrix(
            cameraEyePosition    = [1,0,2], 
            cameraTargetPosition = [0,0,1],
            cameraUpVector       = [0,1,0.5],
            physicsClientId     = self.client
            )

        proj_mat = p.computeProjectionMatrixFOV(
            fov=60,
            aspect=1.0,
            nearVal=0.1,
            farVal=10
            )
                                        
        _, _, rgb, depth, _ = p.getCameraImage(
            width=1200,
            height=1200,
            viewMatrix = view_mat,
            projectionMatrix=proj_mat,
            physicsClientId = self.client
        )
        rgb_arr   = np.array(rgb,  dtype=np.uint8)[:, :, :3]
        depth_arr = np.array(depth, dtype=np.float32)[..., None]

        return rgb_arr, depth_arr

    def _get_proprio(self) -> np.ndarray:
        '''
        * joint_states has length = PANDA_ARM_JOINTS
        * s[0] position value of the joint "s"
        * s[1] velocity value of the joint "s"
        '''
        joint_states = p.getJointStates(
            self.robotId, PANDA_ARM_JOINTS,
            physicsClientId=self.client
        )
        q   = np.array([s[0] for s in joint_states], dtype=np.float32) # joint angle (rad): how much a joint is rotated. Internal configuration.
        dq  = np.array([s[1] for s in joint_states], dtype=np.float32) # velocity
        ee_pos, ee_orn = self._get_ee_pose() # ee_pos: position (x,y,z) of the end effector.
        return np.concatenate([q, dq, ee_pos, ee_orn]).astype(np.float32)
    
    def _get_ee_pose(self):
        state = p.getLinkState(
            self.robotId, self.end_effector,
            physicsClientId=self.client
        )
        return np.array(state[4]), state[5]   # pos, orn (quaternion) in URDF frame of reference.

    # ── reward & termination ─────────────────────────────────────────────────
    def _check_position_success(self):
        return np.sqrt(np.dot(self._get_obs()["ee_pos"] - self.sphereId_position, self._get_obs()["ee_pos"] - self.sphereId_position)) < self.success_threshold

    def _is_done(self):
        if self._check_position_success():
            print(f"End-effector is close (< {self.success_threshold} m) to sphere position")
        return self._check_position_success()

    # ── &&&&&& ─────────────────────────────────────────────────
    @staticmethod
    def _get_end_effector(robotId: int):
        end_effector_index = None

        for i in range(p.getNumJoints(robotId)):
            link_name = p.getJointInfo(robotId, i)[12].decode("utf-8")

            if link_name == "panda_hand":
                end_effector_index = i
                break
        return end_effector_index


In [73]:
arm_env.close()
arm_env = MiniArmEnv()
arm_env.reset()
dt_pos = np.array([0.0,0.0,0.01]) # delta perturbation in the position.
obs, reward, done, info = arm_env.step(action=dt_pos)
print(arm_env._step_count)
print(obs)
print(done)

numActiveThreads = 0
stopping threads
Thread with taskId 0 exiting
Thread TERMINATED
destroy semaphore
semaphore destroyed
destroy main semaphore
main semaphore destroyed
finished
numActiveThreads = 0
btShutDownExampleBrowser stopping threads
Thread with taskId 0 exiting
Thread TERMINATED
destroy semaphore
semaphore destroyed
destroy main semaphore
main semaphore destroyed
startThreads creating 1 threads.
starting thread 0
started thread 0 
argc=2
argv[0] = --unused
argv[1] = --start_demo_name=Physics Server
ExampleBrowserThreadFunc started
X11 functions dynamically loaded using dlopen/dlsym OK!
X11 functions dynamically loaded using dlopen/dlsym OK!
Creating context
Created GL 3.3 context
Direct GLX rendering context obtained
Making context current
GL_VENDOR=Intel
GL_RENDERER=Mesa Intel(R) Graphics (MTL)
GL_VERSION=4.6 (Core Profile) Mesa 23.2.1-1ubuntu3.1~22.04.4
GL_SHADING_LANGUAGE_VERSION=4.60
pthread_getconcurrency()=0
Version = 4.6 (Core Profile) Mesa 23.2.1-1ubuntu3.1~22.04.4
Ve

### C2. Oracle script & C3. Collect a dataset

# TODO: Add lists "pos_EE_list = []" but for rgb, depth, proprio.

In [52]:
class Oracle():
    def reset(self, pos_sphere: np.ndarray):
        '''Adds as attributes to the Oracle object the position of the sphere.
           Store here the target positions. Those that you do not want to change, but rather reached.'''
        self.pos_sphere = pos_sphere.copy()
    
    def act(self, obs: dict):
        current_pos_EE = obs["pos_EE"]
        target = self.pos_sphere
        action = self._delta_to(current_pos_EE, target)
        return action

    def _delta_to(self, current: np.ndarray, target: np.ndarray):
        "Compute the delta position to then move"
        diff = current - target
        dist = np.linalg.norm(diff)
        # normalise the action
        delta = diff / (np.linalg.norm(diff) + 1e-8) # + 1e-8 controls an error arising from np.linalg.norm(diff) = 0.
        return delta

def collect_demos(n_demos: int, out_path: str, domain: str = "source"):
    Path(out_path).parent.mkdir(parents=True, exist_ok=True)
    env = MiniArmEnv(max_steps=400)
    oracle = Oracle()
    
    pos_EE_list = []
    collected = 0

    while collected < n_demos:
        obs = env.reset()
        oracle.reset(obs["pos_sphere"])

        ep_pos_EE = []

        done = False
        while not done:
            action = oracle.act(obs) # generates the action as a delta between current end-effector position and sphere position.
            ep_pos_EE.append(obs["pos_EE"])

            obs, _, done, info = env.step(action)

        if info["success"]:
            pos_EE_list.append(np.stack(ep_pos_EE))
            collected += 1

    env.close()
    _save_hdf5(out_path, pos_EE_list)


def _save_hdf5(out_path: str, pos_EE: list):
    with h5py.File(out_path, "w") as f:
        for i, (ee) in enumerate(zip(pos_EE)):
            grp = f.create_group(f"demo_{i:04d}")
            grp.create_dataset("pos_EE", data=ee, compression="gzip")



In [53]:
collect_demos(n_demos=1, out_path=Path.cwd() / "data" / "demos.hdf5")

startThreads creating 1 threads.
starting thread 0
started thread 0 
argc=2
argv[0] = --unused
argv[1] = --start_demo_name=Physics Server
ExampleBrowserThreadFunc started
X11 functions dynamically loaded using dlopen/dlsym OK!
X11 functions dynamically loaded using dlopen/dlsym OK!
Creating context
Created GL 3.3 context
Direct GLX rendering context obtained
Making context current
GL_VENDOR=Intel
GL_RENDERER=Mesa Intel(R) Graphics (MTL)
GL_VERSION=4.6 (Core Profile) Mesa 23.2.1-1ubuntu3.1~22.04.4
GL_SHADING_LANGUAGE_VERSION=4.60
pthread_getconcurrency()=0
Version = 4.6 (Core Profile) Mesa 23.2.1-1ubuntu3.1~22.04.4
Vendor = Intel
Renderer = Mesa Intel(R) Graphics (MTL)
b3Printf: Selected demo: Physics Server
startThreads creating 1 threads.
starting thread 0
started thread 0 
MotionThreadFunc thread started
ven = Intel
Workaround for some crash in the Intel OpenGL driver on Linux/Ubuntu
ven = Intel
Workaround for some crash in the Intel OpenGL driver on Linux/Ubuntu
End-effector is clos

### C4. Build your dataset class

#### TODO: 
* Open the .hdf5 file
* Explore its content.
* Build dataclass.